In [70]:
import torch

In [71]:
epsilon = 1.0
r0 = 0.1

In [102]:
get_x = lambda: torch.tensor([0.1542, 0.8827, 0.3679, 0.2300, 0.5687, 0.0396, 0.5161, 0.0529, 0.9302,
        0.9654, 0.3532, 0.2273, 0.9690, 0.9541, 0.8267, 0.5761, 0.1615, 0.8755,
        0.6661, 0.7874, 0.7718])
d = get_x().shape[0]
d

21

In [103]:
def value_point(point):
    point = point.reshape(-1,3)
    m = point.shape[0]
    result = 0.0
    r = lambda i, j: torch.linalg.vector_norm(point[i, :]-point[j, :])
    for i in range(m):
      for j in range(i+1,m):
        rij = r0/r(i,j)
        result = result + 4 * epsilon * (rij**12-rij**6)
    return result

x = get_x()
value_point(x)

tensor(-0.5429)

## AD grad

In [109]:
x = get_x().unsqueeze(dim=0)
x.requires_grad_(True)
u = value_point(x).unsqueeze(dim=0).unsqueeze(dim=0)
grad_u = torch.autograd.grad(
    inputs=x,
    outputs=u,
    grad_outputs=torch.ones_like(u),
    create_graph=True,
    retain_graph=True,
)[0]
spatial_laplace_u = []
for i in range(d):
    hess_row = torch.autograd.grad(
        inputs=x,
        outputs=grad_u[:,i].sum(),
        grad_outputs=torch.tensor(1.0),
        create_graph=True,
        retain_graph=True
    )[0]
    spatial_laplace_u.append(hess_row[:,i:i+1])
spatial_laplace_u = torch.cat(spatial_laplace_u, dim=1)
grad_u

tensor([[-1.3169e-03,  3.8306e-03,  3.5812e-03,  7.3281e-04, -3.6869e-03,
         -4.0101e-03, -8.5463e+00, -1.5469e+01,  7.7914e+00,  3.0262e-04,
         -1.4624e-04, -3.0497e-04,  3.2267e-02,  1.7847e-02,  5.9078e-03,
          8.5461e+00,  1.5468e+01, -7.7911e+00, -3.1778e-02, -1.6995e-02,
         -5.5132e-03]], grad_fn=<ViewBackward0>)

In [110]:
x = get_x()
x.requires_grad_(True)
u, vjp_fn = torch.func.vjp(value_point, x)
grad_u = vjp_fn(torch.ones_like(u))
grad_u

(tensor([-1.3169e-03,  3.8306e-03,  3.5812e-03,  7.3281e-04, -3.6869e-03,
         -4.0101e-03, -8.5463e+00, -1.5469e+01,  7.7914e+00,  3.0262e-04,
         -1.4624e-04, -3.0497e-04,  3.2267e-02,  1.7847e-02,  5.9078e-03,
          8.5461e+00,  1.5468e+01, -7.7911e+00, -3.1778e-02, -1.6995e-02,
         -5.5132e-03], grad_fn=<ViewBackward0>),)

## analytic grad

In [76]:
x = get_x()
X = x.reshape(-1,3)
X.shape

torch.Size([7, 3])

In [77]:
m = 7
r = lambda a, b: torch.linalg.vector_norm(a-b)
print("i k l V_der")
for i in range(21):
    k = i // 3
    l = i - 3*k
    assert i == 3*k + l
    V_der = 0.0
    for j in range(m):
        if j == k:
            continue
        r_kj = r(X[k,:], X[j,:])
        term = ( -12 * r0**12 * r_kj**(-14) + 6 * r0**6  * r_kj**(-8) ) * ( X[k,l] - X[j,l] )
        V_der += term
    V_der = 4*epsilon * V_der
    print(i, k, l, V_der)

i k l V_der
0 0 0 tensor(-0.0013)
1 0 1 tensor(0.0038)
2 0 2 tensor(0.0036)
3 1 0 tensor(0.0007)
4 1 1 tensor(-0.0037)
5 1 2 tensor(-0.0040)
6 2 0 tensor(-8.5463)
7 2 1 tensor(-15.4690)
8 2 2 tensor(7.7914)
9 3 0 tensor(0.0003)
10 3 1 tensor(-0.0001)
11 3 2 tensor(-0.0003)
12 4 0 tensor(0.0323)
13 4 1 tensor(0.0178)
14 4 2 tensor(0.0059)
15 5 0 tensor(8.5461)
16 5 1 tensor(15.4681)
17 5 2 tensor(-7.7911)
18 6 0 tensor(-0.0318)
19 6 1 tensor(-0.0170)
20 6 2 tensor(-0.0055)


## AD second der

In [111]:
spatial_laplace_u

tensor([[ 6.3579e-03, -3.1801e-02, -3.7733e-02,  8.2562e-03, -3.2133e-02,
         -3.6738e-02, -1.2835e+00, -3.2841e+02,  2.2986e+01, -1.2844e-03,
         -5.0407e-04, -1.8892e-03, -5.2994e-01, -8.7046e-02,  8.5145e-02,
         -1.2831e+00, -3.2841e+02,  2.2986e+01, -5.3161e-01, -9.3006e-02,
          8.3845e-02]], grad_fn=<CatBackward0>)

In [113]:
m = 7
r = lambda a, b: torch.linalg.vector_norm(a-b)
print("i k l V_der")
for i in range(21):
    k = i // 3
    l = i - 3*k
    assert i == 3*k + l
    V_der = 0.0
    for j in range(m):
        if j == k:
            continue
        r_kj = r(X[k,:], X[j,:])
        term1 = ( 12 * 13 * r0**12 * r_kj**(-14) - 6 * 7 *r0**6  * r_kj**(-8) ) * (( X[k,l] - X[j,l] ) / r_kj)**2 
        term2 = ( - 12 * r0**12 * r_kj**(-14) + 6 * r0**6  * r_kj**(-8) ) * ( 1 - (( X[k,l] - X[j,l] ) / r_kj)**2 )
        term  = term1 + term2
        V_der += term
    V_der = 4*epsilon * V_der
    print(i, k, l, V_der)

i k l V_der
0 0 0 tensor(0.0064)
1 0 1 tensor(-0.0318)
2 0 2 tensor(-0.0377)
3 1 0 tensor(0.0083)
4 1 1 tensor(-0.0321)
5 1 2 tensor(-0.0367)
6 2 0 tensor(-1.2834)
7 2 1 tensor(-328.4093)
8 2 2 tensor(22.9859)
9 3 0 tensor(-0.0013)
10 3 1 tensor(-0.0005)
11 3 2 tensor(-0.0019)
12 4 0 tensor(-0.5299)
13 4 1 tensor(-0.0870)
14 4 2 tensor(0.0851)
15 5 0 tensor(-1.2830)
16 5 1 tensor(-328.4138)
17 5 2 tensor(22.9858)
18 6 0 tensor(-0.5316)
19 6 1 tensor(-0.0930)
20 6 2 tensor(0.0838)


---

## vectorize

In [115]:
X

tensor([[0.1542, 0.8827, 0.3679],
        [0.2300, 0.5687, 0.0396],
        [0.5161, 0.0529, 0.9302],
        [0.9654, 0.3532, 0.2273],
        [0.9690, 0.9541, 0.8267],
        [0.5761, 0.1615, 0.8755],
        [0.6661, 0.7874, 0.7718]])

In [117]:
m = 7
r = lambda a, b: torch.linalg.vector_norm(a-b)

print("i k l V_der")
for k in range(m):
    V_der = 0.0
    for j in range(m):
        if j == k:
            continue
        r_kj = r(X[k,:], X[j,:])
        term = ( -12 * r0**12 * r_kj**(-14) + 6 * r0**6  * r_kj**(-8) ) * ( X[k,:] - X[j,:] )
        V_der += term
    V_der = 4*epsilon * V_der
    print(k, V_der)

#print("i k l V_der")
#for i in range(21):
#    k = i // 3
#    l = i - 3*k
#    assert i == 3*k + l
#    V_der = 0.0
#    for j in range(m):
#        if j == k:
#            continue
#        r_kj = r(X[k,:], X[j,:])
#        term1 = ( 12 * 13 * r0**12 * r_kj**(-14) - 6 * 7 *r0**6  * r_kj**(-8) ) * (( X[k,l] - X[j,l] ) / r_kj)**2 
#        term2 = ( - 12 * r0**12 * r_kj**(-14) + 6 * r0**6  * r_kj**(-8) ) * ( 1 - (( X[k,l] - X[j,l] ) / r_kj)**2 )
#        term  = term1 + term2
#        V_der += term
#    V_der = 4*epsilon * V_der
#    print(i, k, l, V_der)

i k l V_der
0 tensor([-0.0013,  0.0038,  0.0036])
1 tensor([ 0.0007, -0.0037, -0.0040])
2 tensor([ -8.5463, -15.4690,   7.7914])
3 tensor([ 0.0003, -0.0001, -0.0003])
4 tensor([0.0323, 0.0178, 0.0059])
5 tensor([ 8.5461, 15.4681, -7.7911])
6 tensor([-0.0318, -0.0170, -0.0055])
